In [2]:
import os
import pyspark

print("Répertoire :", os.getcwd())
print("Version PySpark :", pyspark.__version__)

Répertoire : /home/jovyan/work
Version PySpark : 3.5.3


# TP : Mise en oeuvre d’un pipeline ETL et d’un Data Lakehouse (Bronze/Silver/Gold)

## Preparation de l'envirronnement

### 1.1 Initialisation de la session Spark

In [3]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Eshop-ETL-Lakehouse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Spark master :", spark.sparkContext.master)
print("Application ID :", spark.sparkContext.applicationId)


Spark version : 3.5.3
Spark master : local[*]
Application ID : local-1794000433002


### 1.2 Ingestion des données sources

In [7]:
customers_raw =(
    spark.read.format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "false")
    .load("data/source/customers.csv")
)

customers_raw.printSchema()
customers_raw.show(10, truncate=False)

print("Nombre de clients :", customers_raw.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C010       |Omar 10      |Ouest |2024-03-11       |
+-----------+-------------+------+-----------------+
only showing top 10 rows

Nombre de clients : 32


In [8]:
# Let's continue the ingestion process by reading the other data sources
def read_csv_data(filename):
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .option("inferSchema", "false")
        .csv(f"data/source/{filename}")
    )

products_raw = read_csv_data("products.csv")
orders_raw = read_csv_data("orders.csv")
order_items_raw = read_csv_data("order_items.csv")


In [9]:
#promotions file
promotions_raw = spark.read.text("data/source/promotions.jsonl")
promotions_raw.printSchema()
promotions_raw.show(truncate=False)

root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
|{"promotion_code":"BAD_JSON","discount_rate":0.10,"start_date":"2025-06-01","end_date":             |
+-----------------------------

In [11]:
# number of lines in each source
sources = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions": promotions_raw
}

for name, df in sources.items():
    print(f"{name}: {df.count()} lignes")


customers: 32 lignes
products: 14 lignes
orders: 44 lignes
order_items: 86 lignes
promotions: 6 lignes
